In [2]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split

from seqeval.metrics import precision_score, recall_score, f1_score, classification_report
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
    set_seed,
)
from datasets import Dataset

d:\Users\DEBJIT DAS\MedBhasha-Resources\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
CSV_PATH = "../dataset/Annotated_Corpus.csv"
MODEL_NAME = "csebuetnlp/banglabert"
OUTPUT_DIR = "./trained_model"

MAX_LENGTH = 256
BATCH_SIZE = 4
EPOCHS = 5
LEARNING_RATE = 2e-5
RANDOM_SEED = 42

set_seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.15.0+cpu
CUDA available: False


In [5]:
ENTITY_TYPES = [
    "DIAGNOSIS", "SYMPTOM", "MEDICATION", "DOSAGE", "TEST",
    "TEST_RESULT", "BODY_SITE", "PROCEDURE", "TEMPORAL",
    "NEGATION_UNCERTAINTY",
]

LABELS = ["O"]
for entity in ENTITY_TYPES:
    LABELS.extend([f"B-{entity}", f"I-{entity}"])

label2id = {label: i for i, label in enumerate(LABELS)}
id2label = {i: label for label, i in label2id.items()}

print("Number of labels:", len(LABELS))
print(LABELS)

Number of labels: 21
['O', 'B-DIAGNOSIS', 'I-DIAGNOSIS', 'B-SYMPTOM', 'I-SYMPTOM', 'B-MEDICATION', 'I-MEDICATION', 'B-DOSAGE', 'I-DOSAGE', 'B-TEST', 'I-TEST', 'B-TEST_RESULT', 'I-TEST_RESULT', 'B-BODY_SITE', 'I-BODY_SITE', 'B-PROCEDURE', 'I-PROCEDURE', 'B-TEMPORAL', 'I-TEMPORAL', 'B-NEGATION_UNCERTAINTY', 'I-NEGATION_UNCERTAINTY']


In [6]:
df = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False)

required_columns = {"note_id", "token", "tag"}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

df = df[["note_id", "token", "tag"]].copy()
for column in ["note_id", "token", "tag"]:
    df[column] = df[column].astype(str).str.strip()

print("Rows:", len(df))
print("Unique notes:", df["note_id"].nunique())
display(df["tag"].value_counts().to_frame("count"))

if (df[["note_id", "token", "tag"]] == "").any().any():
    raise ValueError("Blank note_id, token, or tag values found. Fix the CSV first.")

invalid_tags = sorted(set(df["tag"]) - set(LABELS))
if invalid_tags:
    raise ValueError(f"Unsupported tags found: {invalid_tags}")

Rows: 10743
Unique notes: 534


,count
tag,
O,6881
I-TEMPORAL,709
B-SYMPTOM,530
B-TEMPORAL,423
B-TEST,329
I-TEST,267
B-TEST_RESULT,244
B-NEGATION_UNCERTAINTY,219
B-DIAGNOSIS,169


In [7]:
def check_bio_sequences(dataframe):
    problems = []
    for note_id, group in dataframe.groupby("note_id", sort=False):
        active_entity = None
        for row_number, tag in zip(group.index, group["tag"]):
            if tag == "O":
                active_entity = None
            else:
                prefix, entity_type = tag.split("-", 1)
                if prefix == "B":
                    active_entity = entity_type
                elif prefix == "I" and active_entity != entity_type:
                    problems.append(
                        f"CSV row index {row_number}: {tag} does not continue "
                        f"the active entity in note {note_id!r}"
                    )
                    active_entity = None
    return problems

bio_problems = check_bio_sequences(df)
print("BIO sequence problems:", len(bio_problems))
for problem in bio_problems[:30]:
    print("-", problem)
if bio_problems:
    raise ValueError("Fix BIO sequence problems in the CSV before training.")

BIO sequence problems: 0


In [8]:
notes = []
for note_id, group in df.groupby("note_id", sort=False):
    notes.append({
        "note_id": note_id,
        "tokens": group["token"].tolist(),
        "tags": group["tag"].tolist(),
    })

print("Total notes:", len(notes))
if len(notes) < 10:
    raise ValueError("At least 10 notes are recommended for this split.")

Total notes: 534


In [9]:
train_notes, temp_notes = train_test_split(
    notes, test_size=0.20, random_state=RANDOM_SEED
)
val_notes, test_notes = train_test_split(
    temp_notes, test_size=0.50, random_state=RANDOM_SEED
)

print("Train notes:", len(train_notes))
print("Validation notes:", len(val_notes))
print("Test notes:", len(test_notes))

Train notes: 427
Validation notes: 53
Test notes: 54


In [10]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if not tokenizer.is_fast:
    raise RuntimeError(
        f"{MODEL_NAME} did not load a fast tokenizer. "
        "Choose a compatible fast-tokenizer checkpoint."
    )

def to_hf_dataset(note_list):
    return Dataset.from_dict({
        "tokens": [item["tokens"] for item in note_list],
        "labels": [item["tags"] for item in note_list],
    })

def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
    )
    aligned_labels = []
    for batch_index, word_labels in enumerate(examples["labels"]):
        word_ids = tokenized.word_ids(batch_index=batch_index)
        previous_word_id = None
        label_ids = []
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != previous_word_id:
                label_ids.append(label2id[word_labels[word_id]])
            else:
                label_ids.append(-100)
            previous_word_id = word_id
        aligned_labels.append(label_ids)
    tokenized["labels"] = aligned_labels
    return tokenized

train_dataset = to_hf_dataset(train_notes).map(
    tokenize_and_align_labels, batched=True, remove_columns=["tokens"]
)
val_dataset = to_hf_dataset(val_notes).map(
    tokenize_and_align_labels, batched=True, remove_columns=["tokens"]
)
test_dataset = to_hf_dataset(test_notes).map(
    tokenize_and_align_labels, batched=True, remove_columns=["tokens"]
)

print(train_dataset)

Map: 100%|██████████| 54/54 [00:00<00:00, 6748.28 examples/s]

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 427
})


In [11]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
)
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
print("Model loaded.")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 24182.80it/s]
[transformers] ElectraForTokenClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
classifier.bias                                   | MISSING    | 
classifier.weight                                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded.


In [12]:
def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=2)
    true_predictions, true_labels = [], []

    for prediction, label in zip(predictions, labels):
        pred_tags, true_tags = [], []
        for pred_id, label_id in zip(prediction, label):
            if label_id == -100:
                continue
            pred_tags.append(id2label[int(pred_id)])
            true_tags.append(id2label[int(label_id)])
        true_predictions.append(pred_tags)
        true_labels.append(true_tags)

    return {
        "precision": precision_score(true_labels, true_predictions, zero_division=0),
        "recall": recall_score(true_labels, true_predictions, zero_division=0),
        "f1": f1_score(true_labels, true_predictions, zero_division=0),
    }

In [14]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=EPOCHS,

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,

    learning_rate=LEARNING_RATE,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    logging_steps=10,
    save_total_limit=2,

    report_to="none",

    fp16=torch.cuda.is_available(),

    seed=RANDOM_SEED,
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()

d:\Users\DEBJIT DAS\MedBhasha-Resources\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.611616,0.434352,0.630841,0.671642,0.650602
2,0.260100,0.209645,0.783410,0.845771,0.813397
3,0.155083,0.128166,0.862559,0.905473,0.883495
4,0.111356,0.098735,0.927184,0.950249,0.938575
5,0.083719,0.092670,0.932039,0.955224,0.943489


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.38s/it]
d:\Users\DEBJIT DAS\MedBhasha-Resources\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.87it/s]
d:\Users\DEBJIT DAS\MedBhasha-Resources\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.08s/it]
d:\Users\DEBJIT DAS\MedBhasha-Resources\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00

In [15]:
test_results = trainer.evaluate(
    eval_dataset=test_dataset,
    metric_key_prefix="test"
)
for key, value in test_results.items():
    print(f"{key}: {value}")

d:\Users\DEBJIT DAS\MedBhasha-Resources\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Precision,Recall,F1
0.083719,0.077840,5,0.935484,0.950820,0.943089


test_loss: 0.07783957570791245
test_precision: 0.9354838709677419
test_recall: 0.9508196721311475
test_f1: 0.943089430894309


In [16]:
test_output = trainer.predict(test_dataset)
test_pred_ids = np.argmax(test_output.predictions, axis=2)

true_predictions, true_labels = [], []
for prediction, label in zip(test_pred_ids, test_output.label_ids):
    pred_tags, true_tags = [], []
    for pred_id, label_id in zip(prediction, label):
        if label_id == -100:
            continue
        pred_tags.append(id2label[int(pred_id)])
        true_tags.append(id2label[int(label_id)])
    true_predictions.append(pred_tags)
    true_labels.append(true_tags)

print(classification_report(
    true_labels, true_predictions, digits=4, zero_division=0
))

                      precision    recall  f1-score   support

           BODY_SITE     0.7778    0.8750    0.8235         8
           DIAGNOSIS     1.0000    1.0000    1.0000        18
              DOSAGE     1.0000    1.0000    1.0000        12
          MEDICATION     0.8421    0.8889    0.8649        18
NEGATION_UNCERTAINTY     0.9615    0.9259    0.9434        27
           PROCEDURE     1.0000    0.5714    0.7273         7
             SYMPTOM     0.9344    0.9828    0.9580        58
            TEMPORAL     1.0000    0.9773    0.9885        44
                TEST     0.8750    0.9655    0.9180        29
         TEST_RESULT     0.9167    0.9565    0.9362        23

           micro avg     0.9355    0.9508    0.9431       244
           macro avg     0.9308    0.9143    0.9160       244
        weighted avg     0.9385    0.9508    0.9423       244



In [17]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

with open(os.path.join(OUTPUT_DIR, "labels.json"), "w", encoding="utf-8") as f:
    json.dump(
        {"label2id": label2id, "id2label": id2label},
        f,
        ensure_ascii=False,
        indent=2,
    )

print("Saved model and tokenizer to:", OUTPUT_DIR)

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.25s/it]

Saved model and tokenizer to: ./trained_model


In [19]:
from transformers import pipeline

inference_device = 0 if torch.cuda.is_available() else -1
ner_pipeline = pipeline(
    "token-classification",
    model=OUTPUT_DIR,
    tokenizer=OUTPUT_DIR,
    aggregation_strategy="simple",
    device=inference_device,
)

sample_text = (
    "জ্বর আর কাশি নিয়ে এসেছিলেন রোগী, তাই ডাক্তার আপাতত ৫০০ মিলিগ্রামের প্যারাসিটামল দিয়েছেন।"
)

predictions = ner_pipeline(sample_text)
if not predictions:
    print("No entities detected.")
else:
    for item in predictions:
        print(
            f"Text: {item['word']} | "
            f"Entity: {item['entity_group']} | "
            f"Confidence: {item['score']:.4f}"
        )

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 463.39it/s]


Text: জ্বর | Entity: SYMPTOM | Confidence: 0.9549
Text: কাশি নিয়ে | Entity: SYMPTOM | Confidence: 0.6078
Text: ৫০০ | Entity: MEDICATION | Confidence: 0.2773
Text: মিলিগ্রামের | Entity: DOSAGE | Confidence: 0.4199
Text: প্যারাস | Entity: MEDICATION | Confidence: 0.8978
Text: ##িটা | Entity: MEDICATION | Confidence: 0.3874
Text: ##মল | Entity: DOSAGE | Confidence: 0.3093
Text: দিয়েছেন | Entity: TEST_RESULT | Confidence: 0.2791
